# Coarse MAST3R 几何验证与场景预聚类

## 概述
- **功能**: 对 shortlist 中的候选图像对运行 coarse MAST3R 快速几何验证，构建匹配图，通过连通分量形成初始场景聚类
- **输入**: shortlist（候选图像对） + 原始图片
- **输出**: 场景聚类结果（连通分量列表），每个分量包含图片路径列表 + 内部匹配对列表
- **关键原理**: Shortlist 是候选池，不是最终场景归属；几何验证失败 = 不同场景
- **阈值**: 内点数 τ_inlier ≥ 20，几何一致性 τ_geo ≥ 0.3（可在验证集上调优）

## 1. 环境配置与参数

In [ ]:
import os, sys, json, time, pickle
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from matplotlib import pyplot as plt
import cv2
import networkx as nx

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# ── 路径配置 ──
IMAGE_ROOT = r'../image-matching-challenge-2025/train'
RETRIEVAL_DIR = r'../output/retrieval'
OUTPUT_DIR = r'../output/pre_clustering'
# MAST3R 预训练权重 .pth 文件路径（不是 repo 目录！）
MAST3R_CHECKPOINT = '../models/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 下载 MAST3R 预训练权重（支持断点续传）──
def download_mast3r_checkpoint(url, dest_path, chunk_size=8*1024*1024):
    import urllib.request
    dest_path = os.path.abspath(dest_path)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    req = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(req, timeout=30) as resp:
        total_size = int(resp.headers.get('Content-Length', 0))
    print(f'Remote file size: {total_size / (1024**3):.1f} GB')
    downloaded = 0
    if os.path.exists(dest_path):
        downloaded = os.path.getsize(dest_path)
        if downloaded >= total_size:
            print(f'File already complete: {dest_path}')
            return
        elif downloaded > 0:
            print(f'Resuming from {downloaded / (1024**3):.2f} GB / {total_size / (1024**3):.2f} GB')
    mode = 'ab' if downloaded > 0 else 'wb'
    req = urllib.request.Request(url)
    if downloaded > 0: req.add_header('Range', f'bytes={downloaded}-')
    max_retries = 3
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                with open(dest_path, mode) as f:
                    while True:
                        chunk = resp.read(chunk_size)
                        if not chunk: break
                        f.write(chunk)
                        downloaded += len(chunk)
                        print(f'\r  Downloading... {downloaded / (1024**3):.2f} / {total_size / (1024**3):.2f} GB ({downloaded/total_size*100:.1f}%)', end='', flush=True)
            print(f'\nDownload complete: {dest_path}')
            return
        except Exception as e:
            print(f'\nDownload failed (attempt {attempt+1}/{max_retries}): {e}')
            if attempt < max_retries - 1:
                time.sleep(3)
                if os.path.exists(dest_path): downloaded = os.path.getsize(dest_path)
                mode = 'ab' if downloaded > 0 else 'wb'
                req = urllib.request.Request(url)
                if downloaded > 0: req.add_header('Range', f'bytes={downloaded}-')
            else:
                raise RuntimeError(f'Failed to download after {max_retries} attempts')

if not os.path.exists(MAST3R_CHECKPOINT) or os.path.getsize(MAST3R_CHECKPOINT) < 100 * 1024 * 1024:
    url = 'https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
    if os.path.exists(MAST3R_CHECKPOINT) and os.path.getsize(MAST3R_CHECKPOINT) < 100 * 1024 * 1024:
        os.remove(MAST3R_CHECKPOINT)
    download_mast3r_checkpoint(url, MAST3R_CHECKPOINT)

# ── 几何验证参数 ──
TAU_INLIER = 20              # 内点数量下界
TAU_GEO = 0.3                # 几何一致性分数下界
RANSAC_THRESH = 2.0          # RANSAC 像素阈值
COARSE_IMG_SIZE = 512        # Coarse MAST3R 输入分辨率
BATCH_SIZE = 8               # 批量推理

# ── 孤立图片处理策略 ──
ISOLATED_STRATEGY = 'separate'  # 'separate': 单独成场景; 'discard': 丢弃; 'nearest': 归入最近场景

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 加载 Shortlist

In [ ]:
def load_shortlist(shortlist_path):
    """加载 shortlist JSON"""
    with open(shortlist_path) as f:
        shortlist = json.load(f)
    print(f'Loaded shortlist: {len(shortlist)} images')
    total_pairs = sum(len(v) for v in shortlist.values())
    print(f'  Total candidate pairs: {total_pairs}')
    return shortlist


def load_image_paths(mapping_path):
    """加载图片路径映射"""
    with open(mapping_path) as f:
        mapping = json.load(f)
    print(f'Loaded {len(mapping)} image paths')
    return mapping


shortlist = load_shortlist(os.path.join(RETRIEVAL_DIR, 'shortlist.json'))
path_mapping = load_image_paths(os.path.join(RETRIEVAL_DIR, 'image_paths.json'))

# 构建去重后的候选对列表
candidate_pairs = set()
for img_name, candidates in shortlist.items():
    for c in candidates:
        pair = tuple(sorted([img_name, c['name']]))
        candidate_pairs.add(pair)

candidate_pairs = list(candidate_pairs)
print(f'\nUnique candidate pairs: {len(candidate_pairs)}')

## 3. 加载 Coarse MAST3R 模型

使用 MAST3R 的 coarse 模式进行快速几何验证。与训练时的 backbone 复用同一架构，但使用预训练权重做推理。

In [ ]:
sys.path.insert(0, '../mast3r')
from mast3r.model import AsymmetricMASt3R


class CoarseMASt3R(nn.Module):
    """
    Coarse MAST3R：用于快速几何验证。
    输入图像对 → 输出匹配质量分数（内点数量 + 几何一致性）。
    """

    def __init__(self, checkpoint_path):
        super().__init__()
        # from_pretrained 接受 .pth 文件路径（注意：必须是文件路径，不是目录）
        self.model = AsymmetricMASt3R.from_pretrained(checkpoint_path)

    def forward(self, view1, view2):
        """
        Args:
            view1, view2: dict with 'img' (B, 3, H, W) and 'instance'
        Returns:
            (view1_pred, view2_pred) tuple of dicts
        """
        return self.model(view1, view2)


coarse_mast3r = CoarseMASt3R(MAST3R_CHECKPOINT).to(device).eval()
print('Coarse MAST3R loaded.')

# 图像预处理
coarse_transform = transforms.Compose([
    transforms.Resize(int(COARSE_IMG_SIZE * 1.14)),
    transforms.CenterCrop(COARSE_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 4. 几何验证函数

对每对图像运行 coarse MAST3R，计算：
1. **内点数量（num_inliers）**: 双向置信度都高的匹配点数
2. **几何一致性分数（geo_consistency）**: 3D 点图之间的一致性
3. 两个指标都超过阈值才保留该边。

In [ ]:
@torch.no_grad()
def geometric_verification(model, img1_path, img2_path, transform):
    """
    对一对图像进行几何验证

    Args:
        model: CoarseMASt3R
        img1_path, img2_path: 图像路径
        transform: 预处理 transform
    Returns:
        verified: bool — 是否通过验证
        num_inliers: int — 内点数量
        geo_score: float — 几何一致性分数
        info: dict — 额外信息
    """
    # 加载图片
    img1 = Image.open(img1_path).convert('RGB')
    img2 = Image.open(img2_path).convert('RGB')

    # 预处理
    t1 = transform(img1).unsqueeze(0).to(device)
    t2 = transform(img2).unsqueeze(0).to(device)

    # MAST3R 推理 — 直接传入两个 view dict
    view1 = {'img': t1, 'instance': []}
    view2 = {'img': t2, 'instance': []}
    res1, res2 = model(view1, view2)

    # 从输出提取几何信息
    conf1 = res1.get('conf')  # (B, H, W) confidence map
    conf2 = res2.get('conf')
    pointmap1 = res1.get('pts3d')  # (B, H, W, 3) 3D coordinates
    pointmap2 = res2.get('pts3d_in_other_view')  # view2's pts3d in view1's frame

    if conf1 is None or conf2 is None:
        return False, 0, 0.0, {'error': 'No confidence output'}

    # ── 计算内点数量 ──
    # 高置信度像素（双向）
    conf_threshold = 0.5
    mask1 = conf1.squeeze() > conf_threshold
    mask2 = conf2.squeeze() > conf_threshold
    mutual_mask = mask1 & mask2
    num_inliers = mutual_mask.sum().item()

    # ── 计算几何一致性 ──
    # 比较两个点图在重叠区域的 3D 坐标一致性
    geo_score = 0.0
    if pointmap1 is not None and pointmap2 is not None and num_inliers > 0:
        pm1 = pointmap1.squeeze()  # (H, W, 3)
        pm2 = pointmap2.squeeze()

        # 取高置信度区域的 3D 点
        pts1 = pm1[mutual_mask]  # (N, 3)
        pts2 = pm2[mutual_mask]

        if len(pts1) > 3:
            # 计算尺度对齐后的相对误差
            dist = torch.norm(pts1 - pts2, dim=-1)            # (N,)
            scale1 = torch.norm(pts1, dim=-1).mean()
            scale2 = torch.norm(pts2, dim=-1).mean()
            scale_factor = (scale1 + scale2) / 2 + 1e-6
            normalized_error = (dist / scale_factor).mean()
            geo_score = float(1.0 / (1.0 + normalized_error))

    # ── 判断是否通过验证 ──
    verified = num_inliers >= TAU_INLIER and geo_score >= TAU_GEO

    return verified, num_inliers, geo_score, {}


def estimate_fundamental_matrix(kpts1, kpts2, ransac_thresh=RANSAC_THRESH):
    """
    用关键点估算基础矩阵作为补充几何验证。
    如果无法计算有效基础矩阵 → 不同场景。
    """
    if len(kpts1) < 8 or len(kpts2) < 8:
        return False, 0, None
    try:
        F, mask = cv2.findFundamentalMat(
            kpts1, kpts2, cv2.FM_RANSAC,
            ransac_thresh, 0.99
        )
        if F is None or F.shape[0] != 3:
            return False, 0, None
        num_inliers = int(mask.sum())
        return num_inliers >= TAU_INLIER, num_inliers, F
    except Exception:
        return False, 0, None


print('Geometric verification functions ready.')

## 5. 批量几何验证

对所有候选对运行 coarse MAST3R 几何验证。使用批处理提高效率。

In [ ]:
print(f'Running geometric verification on {len(candidate_pairs)} pairs...')
t0 = time.time()

verified_edges = []     # 通过验证的边
verification_results = {}  # 所有验证结果
isolated_images = set(shortlist.keys())  # 初始全部为孤立，通过验证后移除

for name1, name2 in tqdm(candidate_pairs, desc='Geometric verification'):
    p1 = path_mapping.get(name1)
    p2 = path_mapping.get(name2)
    if p1 is None or p2 is None:
        continue

    verified, num_inliers, geo_score, info = geometric_verification(
        coarse_mast3r, p1, p2, coarse_transform
    )

    pair_key = f'{name1}::{name2}'
    verification_results[pair_key] = {
        'verified': verified,
        'num_inliers': num_inliers,
        'geo_score': geo_score,
    }

    if verified:
        verified_edges.append((name1, name2, num_inliers, geo_score))
        isolated_images.discard(name1)
        isolated_images.discard(name2)

t1 = time.time()
print(f'\nVerification completed in {t1-t0:.1f}s')
print(f'  Verified edges: {len(verified_edges)} / {len(candidate_pairs)} ({100*len(verified_edges)/len(candidate_pairs):.1f}%)')
print(f'  Isolated images: {len(isolated_images)}')

## 6. 构建匹配图 & 连通分量

- 节点 = 图片
- 边 = 通过几何验证的图像对（权重 = 几何一致性分数）
- 连通分量算法（BFS/DFS/Union-Find）将图分割为场景簇

In [ ]:
# ── 构建图 ──
G = nx.Graph()

# 添加所有图片作为节点
all_images = list(shortlist.keys())
G.add_nodes_from(all_images)

# 添加通过验证的边
for name1, name2, num_inliers, geo_score in verified_edges:
    G.add_edge(name1, name2, num_inliers=num_inliers, geo_score=geo_score)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# ── 连通分量 ──
connected_components = list(nx.connected_components(G))
print(f'\nConnected components (scenes): {len(connected_components)}')

# 统计各分量大小
component_sizes = sorted([len(cc) for cc in connected_components], reverse=True)
print(f'Component sizes: {component_sizes[:20]}...' if len(component_sizes) > 20 else f'Component sizes: {component_sizes}')

# ── 处理孤立图片 ──
if ISOLATED_STRATEGY == 'separate':
    # 孤立图片各自独立成一个场景
    for img in isolated_images:
        connected_components.append({img})
    print(f'Added {len(isolated_images)} isolated images as separate scenes')
elif ISOLATED_STRATEGY == 'discard':
    print(f'Discarded {len(isolated_images)} isolated images')
elif ISOLATED_STRATEGY == 'nearest':
    # 将孤立图片归入相似度最高的邻接场景
    for img in isolated_images:
        candidates = shortlist.get(img, [])
        for c in sorted(candidates, key=lambda x: x['score'], reverse=True):
            # 找到 c['name'] 所属的连通分量
            for cc in connected_components:
                if c['name'] in cc:
                    cc.add(img)
                    break
            else:
                continue
            break
    print(f'Assigned {len(isolated_images)} isolated images to nearest scenes')

## 7. 场景聚类结果保存

每个连通分量保存为：
- 图片路径列表
- 分量内部的匹配对列表
- 分量统计信息（大小、平均几何分数等）

In [ ]:
scene_clusters = []

for i, component in enumerate(connected_components):
    images = sorted(component)

    # 收集分量内部的匹配对
    internal_edges = []
    for name1, name2, num_inliers, geo_score in verified_edges:
        if name1 in component and name2 in component:
            internal_edges.append({
                'img1': name1,
                'img2': name2,
                'num_inliers': num_inliers,
                'geo_score': geo_score,
            })

    avg_geo = np.mean([e['geo_score'] for e in internal_edges]) if internal_edges else 0.0

    cluster_info = {
        'scene_id': i,
        'num_images': len(images),
        'images': images,
        'image_paths': [path_mapping.get(name, '') for name in images],
        'internal_edges': internal_edges,
        'num_internal_edges': len(internal_edges),
        'is_isolated': len(images) == 1 and images[0] in isolated_images,
        'avg_geo_score': float(avg_geo),
    }
    scene_clusters.append(cluster_info)

# 按场景大小排序
scene_clusters.sort(key=lambda x: x['num_images'], reverse=True)

# 保存
output_path = os.path.join(OUTPUT_DIR, 'scene_clusters.json')
with open(output_path, 'w') as f:
    json.dump(scene_clusters, f, indent=2)
print(f'Scene clusters saved to {output_path}')

# 保存验证边列表（用于后续匹配）
edges_output = os.path.join(OUTPUT_DIR, 'verified_edges.json')
with open(edges_output, 'w') as f:
    json.dump([{
        'img1': e[0], 'img2': e[1],
        'num_inliers': e[2], 'geo_score': e[3],
    } for e in verified_edges], f, indent=2)
print(f'Verified edges saved to {edges_output}')

## 8. 可视化

场景聚类统计和匹配图可视化。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 场景大小分布
ax1 = axes[0]
sizes = [c['num_images'] for c in scene_clusters if not c['is_isolated']]
if sizes:
    ax1.bar(range(len(sizes)), sorted(sizes, reverse=True), color='steelblue', edgecolor='white')
ax1.set_xlabel('Scene rank')
ax1.set_ylabel('Number of images')
ax1.set_title(f'Scene Size Distribution\n({len(sizes)} scenes, excl. isolated)')
ax1.grid(axis='y', alpha=0.3)

# 2. 几何分数分布
ax2 = axes[1]
geo_scores = [e['geo_score'] for e in scene_clusters for e2 in e['internal_edges']]
if geo_scores:
    ax2.hist(geo_scores, bins=50, color='coral', edgecolor='white', alpha=0.8)
    ax2.axvline(x=TAU_GEO, color='red', linestyle='--', label=f'Threshold ({TAU_GEO})')
    ax2.legend()
ax2.set_xlabel('Geometric consistency score')
ax2.set_ylabel('Frequency')
ax2.set_title('Geometric Score Distribution')

# 3. 内点数分布
ax3 = axes[2]
inlier_counts = [e['num_inliers'] for e in scene_clusters for e2 in e['internal_edges']]
if inlier_counts:
    ax3.hist(inlier_counts, bins=50, color='seagreen', edgecolor='white', alpha=0.8)
    ax3.axvline(x=TAU_INLIER, color='red', linestyle='--', label=f'Threshold ({TAU_INLIER})')
    ax3.legend()
ax3.set_xlabel('Number of inliers')
ax3.set_ylabel('Frequency')
ax3.set_title('Inlier Count Distribution')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'clustering_analysis.png'), dpi=150)
plt.show()

## 9. 匹配图可视化（采样）

如果图太大则采样最大的几个分量进行可视化。

In [ ]:
def visualize_matching_graph(G, component_nodes, title, ax):
    """可视化单个连通分量的匹配图"""
    sub_g = G.subgraph(component_nodes)
    pos = nx.spring_layout(sub_g, seed=42, k=2)
    nx.draw_networkx_nodes(sub_g, pos, ax=ax, node_size=50, node_color='steelblue', alpha=0.8)
    nx.draw_networkx_edges(sub_g, pos, ax=ax, alpha=0.3, width=0.5)
    ax.set_title(title)
    ax.axis('off')


# 可视化最大的几个分量
n_show = min(4, len(scene_clusters))
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    cc_nodes = scene_clusters[i]['images']
    if len(cc_nodes) > 50:
        cc_nodes = cc_nodes[:50]  # 采样
    visualize_matching_graph(
        G, cc_nodes,
        f"Scene {scene_clusters[i]['scene_id']}\n({scene_clusters[i]['num_images']} images)",
        axes[i]
    )

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'matching_graphs.png'), dpi=150)
plt.show()

## 10. 输出汇总

预聚类完成，结果供后续 COLMAP 重建使用。

In [ ]:
print('=' * 60)
print('Phase 3 Complete: Pre-clustering')
print('=' * 60)
print(f'Output directory: {OUTPUT_DIR}')
print(f'  - scene_clusters.json   : {len(scene_clusters)} scene clusters')
print(f'  - verified_edges.json   : {len(verified_edges)} verified edges')
print(f'  - clustering_analysis.png : statistics')
print(f'  - matching_graphs.png   : graph visualization')
print(f'\nScene statistics:')
print(f'  Total scenes:           {len(scene_clusters)}')
print(f'  Multi-image scenes:     {sum(1 for c in scene_clusters if c["num_images"] > 1)}')
print(f'  Isolated images:        {sum(1 for c in scene_clusters if c["is_isolated"])}')
print(f'  Largest scene:          {scene_clusters[0]["num_images"]} images')
print(f'  Avg images/scene:       {np.mean([c["num_images"] for c in scene_clusters]):.1f}')
print(f'\nNext steps:')
print(f'  1. Keypoint extraction (if not done) → feature_retrieval_shortlist.ipynb')
print(f'  2. MAST3R dense matching → mast3r_matching.ipynb')
print(f'  3. COLMAP reconstruction per scene → colmap_reconstruction.ipynb')